# Randomized Property Testing for LLM-Generated Programs
### Prototype: the *sorting* program class

**Goal.** Given a program `f` (e.g., written by an LLM) that claims to sort a list, decide with few random queries whether it is correct.

**Plan of this notebook**
1. Define the property and the testers (theory in brief)
2. Build a set of "LLM-style" buggy sorting programs
3. Compare testers: uniform random testing, randomized property tester, structured-distribution tester, Hypothesis
4. Compare empirical detection with the theoretical bound `1 - (1-p)^k`
5. Next steps (plug in real LLM outputs)

Run the cells top to bottom. Runtime is about 1-3 minutes.

In [ ]:
!pip -q install hypothesis

In [ ]:
import random, math, time, copy, zlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from hypothesis import given, settings, strategies as st, Phase, HealthCheck

SEED = 42
random.seed(SEED); np.random.seed(SEED)

## 1. Property and testers

**Property $\Pi$ (sorting):** for every input list $x$, the output $y=f(x)$ satisfies
1. **Sortedness:** $y_1 \le y_2 \le \dots \le y_n$
2. **Permutation:** $y$ has the same multiset of elements as $x$

**Oracle-free permutation check (randomized fingerprint).**
Pick a prime $P=2^{61}-1$ and a random $r\in\mathbb{Z}_P$. Compare
$$H(x)=\prod_i (r - x_i) \bmod P \quad\text{vs}\quad H(y).$$
If the multisets differ, then $H(x)-H(y)$ is a non-zero polynomial of degree $\le n$ in $r$, so by Schwartz-Zippel
$$\Pr[\text{false accept}] \le n/P \approx 0$$
This costs $O(n)$ per query and needs no reference implementation.

**Query-complexity bound.** If a bug makes $f$ fail on a fraction $p$ of inputs drawn from distribution $D$, then $k$ independent queries all pass with probability $(1-p)^k \le e^{-pk}$. To reach false-acceptance $\le\delta$ we need
$$k \ge \frac{\ln(1/\delta)}{p}.$$
So the **input distribution decides $p$**. Uniform inputs make many realistic bugs (duplicates, empty list, long lists) very rare, which is why a structured distribution can need far fewer queries.

In [ ]:
P = (1 << 61) - 1

def fingerprint(arr, r):
    h = 1
    for a in arr:
        h = (h * ((r - a) % P)) % P
    return h

def check_property(x, y, rng):
    """Oracle-free check of the sorting property. Returns True if (x, y) passes."""
    if not isinstance(y, list) or len(y) != len(x):
        return False
    for i in range(len(y) - 1):                  # sortedness
        if y[i] > y[i + 1]:
            return False
    r = rng.randrange(P)                          # permutation via fingerprint
    return fingerprint(x, r) == fingerprint(y, r)

## 2. Input distributions

* **Uniform:** length $\in[0,50]$, values uniform in $[-10^6,10^6]$ (what naive random testing typically does)
* **Structured:** a mixture that deliberately includes empty/singleton lists, small value ranges (many duplicates), negatives, already-sorted / reverse-sorted lists, and long lists

In [ ]:
def gen_uniform(rng):
    n = rng.randint(0, 50)
    return [rng.randint(-10**6, 10**6) for _ in range(n)]

def gen_structured(rng):
    kind = rng.choice(["empty", "single", "small_range", "dups", "neg", "sorted", "reverse", "long", "mixed"])
    if kind == "empty":   return []
    if kind == "single":  return [rng.randint(-100, 100)]
    if kind == "small_range":
        return [rng.randint(0, 3) for _ in range(rng.randint(2, 15))]
    if kind == "dups":
        v = rng.randint(-5, 5)
        return [v if rng.random() < 0.5 else rng.randint(-5, 5) for _ in range(rng.randint(3, 20))]
    if kind == "neg":
        return [rng.randint(-1000, -1) for _ in range(rng.randint(2, 20))]
    if kind == "sorted":
        return sorted(rng.randint(-100, 100) for _ in range(rng.randint(2, 30)))
    if kind == "reverse":
        return sorted((rng.randint(-100, 100) for _ in range(rng.randint(2, 30))), reverse=True)
    if kind == "long":
        return [rng.randint(-10**6, 10**6) for _ in range(rng.randint(50, 200))]
    return [rng.randint(-50, 50) for _ in range(rng.randint(0, 40))]

## 3. Program under test: a correct version and "LLM-style" bugs

These imitate bug patterns often seen in generated code (off-by-one, edge cases, wrong handling of duplicates or negatives, silent truncation). 
*Later you can replace this section with real LLM outputs (see Section 7).*

In [ ]:
def sort_correct(a):
    return sorted(a)

def bug_drop_duplicates(a):          # "dedupe" mistake
    return sorted(set(a))

def bug_drop_last(a):                # off-by-one: loses the last element
    return sorted(a[:-1]) if a else []

def bug_empty_crash(a):              # edge case: crashes on empty list
    m = a[0]
    return sorted(a)

def bug_truncate_long(a):            # only sorts the first 40 elements, rest appended unsorted
    return sorted(a[:40]) + a[40:] if len(a) > 40 else sorted(a)

def bug_abs_order(a):                # wrong with negatives: orders by absolute value
    return sorted(a, key=abs)

def bug_wrong_on_triple(a):          # rare: breaks when some value appears >= 3 times
    from collections import Counter
    if a and max(Counter(a).values()) >= 3:
        return sorted(a)[::-1]
    return sorted(a)

def bug_bubble_early_stop(a):        # bubble sort with a bad loop bound (stops one pass early)
    a = list(a)
    n = len(a)
    for i in range(max(0, n - 2)):   # should be n-1 passes
        for j in range(n - 1 - i):
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
    return a

PROGRAMS = {
    "correct": sort_correct,
    "drop_duplicates": bug_drop_duplicates,
    "drop_last": bug_drop_last,
    "empty_crash": bug_empty_crash,
    "truncate_long (n>40)": bug_truncate_long,
    "abs_order (negatives)": bug_abs_order,
    "triple_value": bug_wrong_on_triple,
    "bubble_early_stop": bug_bubble_early_stop,
}
BUGGY = [k for k in PROGRAMS if k != "correct"]

## 4. Testers

Each tester returns the **number of queries until the first failure is found**, or `None` if no failure is found within the budget.

| Tester | Input distribution | Checks outputs with |
|---|---|---|
| `oracle_uniform` | uniform | reference `sorted()` |
| `property_uniform` | uniform | sortedness + fingerprint (no oracle) |
| `property_structured` | structured | sortedness + fingerprint (no oracle) |
| `hypothesis` | Hypothesis `lists(integers())` | sortedness + permutation (via `Counter`) |

In [ ]:
from collections import Counter

def call_safely(f, x):
    try:
        return f(list(x)), None
    except Exception as e:
        return None, e

def tester_oracle_uniform(f, k, rng):
    for q in range(1, k + 1):
        x = gen_uniform(rng)
        y, err = call_safely(f, x)
        if err is not None or y != sorted(x):
            return q
    return None

def tester_property(gen):
    def run(f, k, rng):
        for q in range(1, k + 1):
            x = gen(rng)
            y, err = call_safely(f, x)
            if err is not None or not check_property(x, y, rng):
                return q
        return None
    return run

def tester_hypothesis(f, k, rng):
    calls = {"n": 0}
    @settings(max_examples=k, phases=[Phase.generate], database=None, deadline=None,
              suppress_health_check=list(HealthCheck), derandomize=False)
    @given(st.lists(st.integers(-10**6, 10**6), max_size=50))
    def prop(x):
        calls["n"] += 1
        y, err = call_safely(f, x)
        assert err is None and isinstance(y, list)
        assert all(y[i] <= y[i+1] for i in range(len(y)-1))
        assert Counter(y) == Counter(x)
    try:
        prop()
        return None
    except AssertionError:
        return calls["n"]
    except Exception:
        return calls["n"]

TESTERS = {
    "oracle_uniform": tester_oracle_uniform,
    "property_uniform": tester_property(gen_uniform),
    "property_structured": tester_property(gen_structured),
}

## 5. Experiment A - estimating the failure rate $p$ of each bug

For each bug we estimate $p$ = probability that a random input from each distribution exposes it. This is the quantity that controls the theory: the expected number of queries is about $1/p$.

In [ ]:
def failure_rate(f, gen, n=20000, seed=1):
    rng = random.Random(seed)
    bad = 0
    for _ in range(n):
        x = gen(rng)
        y, err = call_safely(f, x)
        if err is not None or y != sorted(x):
            bad += 1
    return bad / n

rows = []
for name in BUGGY:
    pu = failure_rate(PROGRAMS[name], gen_uniform)
    ps = failure_rate(PROGRAMS[name], gen_structured)
    rows.append({"bug": name, "p_uniform": pu, "p_structured": ps,
                 "k_needed_uniform (delta=0.01)": math.ceil(math.log(100)/pu) if pu > 0 else float("inf"),
                 "k_needed_structured (delta=0.01)": math.ceil(math.log(100)/ps) if ps > 0 else float("inf")})
df_p = pd.DataFrame(rows).set_index("bug")
df_p.round(4)

## 6. Experiment B - detection probability vs number of queries

For each (bug, tester) we run many independent trials and record the query index at which the bug is first found. Then
`Pr[detect within k queries]` = fraction of trials with detection index $\le k$.

We also check **false rejects** on the correct program (must be 0).

In [ ]:
KMAX = 300
TRIALS = 300          # trials for our own testers
HYP_TRIALS = 40       # Hypothesis is slower, so fewer trials
KS = [1, 2, 5, 10, 20, 50, 100, 200, 300]

def run_trials(tester, f, trials, kmax, seed):
    rng = random.Random(seed)
    first = []
    for _ in range(trials):
        first.append(tester(f, kmax, rng))
    return first

def detect_curve(first, ks):
    n = len(first)
    return [sum(1 for q in first if q is not None and q <= k) / n for k in ks]

results = {}   # (bug, tester) -> list of first-detection indices
t0 = time.time()
for bug in BUGGY + ["correct"]:
    for tname, tester in TESTERS.items():
        results[(bug, tname)] = run_trials(tester, PROGRAMS[bug], TRIALS, KMAX, seed=zlib.crc32(f"{bug}|{tname}".encode()))
    results[(bug, "hypothesis")] = run_trials(tester_hypothesis, PROGRAMS[bug], HYP_TRIALS, KMAX, seed=0)
print(f"done in {time.time()-t0:.1f}s")

# false rejects on the correct program
for tname in list(TESTERS) + ["hypothesis"]:
    fr = sum(1 for q in results[("correct", tname)] if q is not None)
    print(f"false rejects on correct program | {tname:20s}: {fr}")

In [ ]:
tester_names = list(TESTERS) + ["hypothesis"]
table = []
for bug in BUGGY:
    row = {"bug": bug}
    for tname in tester_names:
        first = results[(bug, tname)]
        row[f"{tname} P(detect<=20)"] = detect_curve(first, [20])[0]
    table.append(row)
df_20 = pd.DataFrame(table).set_index("bug")
df_20.round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharey=True)
colors = {"oracle_uniform": "tab:blue", "property_uniform": "tab:orange",
          "property_structured": "tab:green", "hypothesis": "tab:red"}
for ax, bug in zip(axes.ravel(), BUGGY):
    for tname in tester_names:
        ax.plot(KS, detect_curve(results[(bug, tname)], KS), marker="o", label=tname, color=colors[tname])
    pu = df_p.loc[bug, "p_uniform"]
    if pu > 0:
        ax.plot(KS, [1 - (1 - pu) ** k for k in KS], "k--", label="theory 1-(1-p)^k (uniform)")
    ax.set_xscale("log"); ax.set_title(bug); ax.set_xlabel("queries k"); ax.grid(alpha=.3)
axes[0, 0].set_ylabel("P(detect within k)"); axes[1, 0].set_ylabel("P(detect within k)")
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**How to read the plots**
* The dashed black line is the theoretical curve for the uniform distribution. The `oracle_uniform` and `property_uniform` curves should follow it closely, which supports the $(1-p)^k$ analysis.
* The structured tester should rise much faster for the rare bugs (`drop_duplicates`, `empty_crash`, `triple_value`, `truncate_long`).
* `property_uniform` matches `oracle_uniform` in detection power: the oracle-free check loses nothing for sorting and needs **no reference solution**, which is the realistic setting for LLM-generated code (usually no trusted implementation exists).
* `hypothesis` is strong on some bugs but weak on rare ones (e.g., `truncate_long`, `triple_value`) because its default distribution rarely produces long lists or repeated values.

In [ ]:
# Efficiency: time per query (same inputs) for oracle check vs property check
rng = random.Random(0)
xs = [[rng.randint(-10**6, 10**6) for _ in range(20000)] for _ in range(20)]
ys = [sorted(x) for x in xs]

t = time.time()
for x, y in zip(xs, ys): assert y == sorted(x)
t_oracle = (time.time() - t) / len(xs)

t = time.time()
for x, y in zip(xs, ys): assert check_property(x, y, rng)
t_prop = (time.time() - t) / len(xs)
print(f"oracle check   : {t_oracle*1000:.2f} ms/query (n=20000)")
print(f"property check : {t_prop*1000:.2f} ms/query (n=20000)")
print("Note: asymptotically the property check is O(n) vs O(n log n) for sorting, but in pure Python the built-in sorted() is C-fast, so the oracle can be faster here. The real advantage is that no reference solution is needed.")

## 7. Next steps: use real LLM-generated programs

To move from simulated bugs to real ones:

1. Write ~10-20 prompts of the form *"Write a Python function `my_sort(a)` that sorts a list of integers"* (vary the wording, e.g. ask for stability, in-place, handling duplicates).
2. Generate 5-10 samples per prompt with an LLM (ChatGPT, Claude, Gemini, or an open model through `transformers`).
3. Paste each function into a dict like `PROGRAMS` (execute with `exec` inside a sandbox with a timeout; never run unreviewed code with side effects).
4. Establish ground truth with a very large test set (for example 10^5 inputs against `sorted`).
5. Rerun Experiments A and B and compare.

Then extend to a **second class** (e.g., an algebraic property such as linearity $f(x+y)=f(x)+f(y)$, or a round-trip property `decode(encode(x)) == x`) and add the formal proofs.

In [ ]:
# Template: load an LLM-written function from source text (edit/extend as needed)
import multiprocessing as mp

def load_function(source, fname):
    ns = {}
    exec(source, ns)       # WARNING: run only code you have reviewed
    return ns[fname]

example_src = '''
def my_sort(a):
    # (paste LLM output here)
    return sorted(a)
'''
f_llm = load_function(example_src, "my_sort")
print("first failure at query:", tester_property(gen_structured)(f_llm, 200, random.Random(0)))